In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df=spark.read.format('delta')\
    .load('abfss://bronze@adventwork.dfs.core.windows.net/customers')

### **Drop Rescued Data Column**

In [0]:
df=df.drop("_rescued_data")

### **Checking Schema**

In [0]:
df.printSchema()

root
 |-- CustomerKey: string (nullable = true)
 |-- Prefix: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- BirthDate: string (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- EmailAddress: string (nullable = true)
 |-- AnnualIncome: string (nullable = true)
 |-- TotalChildren: string (nullable = true)
 |-- EducationLevel: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- HomeOwner: string (nullable = true)



### **Fixing BirtDate column's format**

In [0]:
df=df.withColumn('BirthDate',regexp_replace(col("BirthDate"),'/','-'))

In [0]:
df=df.withColumn('BirthDate',to_date(col("BirthDate"),'m-d-yyyy'))

### **Fixing TotalChildren column's format**

In [0]:
df=df.withColumn('TotalChildren',col("TotalChildren").cast('int'))

### **Fixing CustomerKey column's format**

In [0]:
df=df.withColumn('CustomerKey',col("CustomerKey").cast('int'))

### **InitCap for Prefix , FirstName , LastName columns**

In [0]:
df=df.withColumn('Prefix',initcap(col("Prefix")))\
     .withColumn('FirstName',initcap(col("FirstName")))\
     .withColumn('LastName',initcap(col("LastName")))

### **Fixing Frefix column's values**

In [0]:
df=df.withColumn('Prefix',regexp_replace(col("Prefix"),'Ms.','Mrs.'))

### **Changing AnnualIncome column's logic**

In [0]:
df=df.withColumn('Currency',col('AnnualIncome').substr(1,1))

In [0]:
df=df.withColumn('AnnualIncome',substr(col("AnnualIncome"),lit(2),length(col("AnnualIncome"))))

In [0]:
df=df.withColumn('AnnualIncome',regexp_replace(col("AnnualIncome"),',',''))

In [0]:
df=df.withColumn('AnnualIncome',col("AnnualIncome").cast('int'))

### **Trying Fix NULL values in Gender and Prefix Columns**

In [0]:
df.filter((col("Prefix").isNull()) & ((col('Gender').isNotNull()) & ~col("Gender").isin('NA'))).display()

CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner,Currency


In [0]:
df.filter((col("Gender").isNull()) & ((col('Prefix').isNotNull()) & ~col("Prefix").isin('NA'))).display()

CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner,Currency


### **Filling NULLS**

In [0]:
df=df.fillna('NA',subset=["Prefix",'MaritalStatus','Gender','Occupation'])

### **Checking data quality for EmailAddress**

In [0]:
email_regex = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

df=df.filter(col("EmailAddress").rlike(email_regex))

### **Writing invalid data to quarantine table**

In [0]:
df_invalid=df.filter(
    (~col("Prefix").isin(['Mr.','Mrs.','NA'])) |
    (~col('MaritalStatus').isin(['M','S','NA'])) |
    (~col("Gender").isin(['M','F','NA'])) |
    (~col("Occupation").isin(['Professional','Management','Skilled Manual','Clerical','Manual','NA'])) |
    (col("TotalChildren") < 0) |
    (~col("EducationLevel").isin(['Bachelors','Partial College','High School','Partial High School','Graduate Degree','NA']))

    )

In [0]:
df_invalid.write.format('delta').mode('overwrite')\
    .save('abfss://silver@adventwork.dfs.core.windows.net/quarantine/customers')

### **Data Writing**

In [0]:
if spark.catalog.tableExists('adventure_works.silver.customers'):
    df_silver_customers = spark.read.table('adventure_works.silver.customers')
    df = df.join(df_silver_customers , ['CustomerKey'] , "left_anti")

In [0]:
df.write.format('delta')\
    .mode('append')\
    .save('abfss://silver@adventwork.dfs.core.windows.net/customers')

In [0]:
%sql
create table if not exists adventure_works.silver.customers
using delta
location 'abfss://silver@adventwork.dfs.core.windows.net/customers'